# 05. Monitoramento de Data Drift (PSI)
Neste caderno simulamos a passagem do tempo de Abril a Setembro. O modelo foi treinado com o perfil de Q1 (Jan-Mar). Em Maio, ocorre uma anomalia (drástico aumento no tempo de fila). Vamos observar o PSI acusando a degradação do modelo.

In [ ]:
%pip install -q xgboost scikit-learn mlflow


In [ ]:
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql.functions import col

def calculate_psi(expected, actual, buckets=10):
    breaks = np.linspace(0, 1, buckets + 1)
    expected_perc = np.histogram(expected, breaks)[0] / len(expected)
    actual_perc = np.histogram(actual, breaks)[0] / len(actual)
    
    expected_perc = np.where(expected_perc == 0, 0.0001, expected_perc)
    actual_perc = np.where(actual_perc == 0, 0.0001, actual_perc)
    
    psi_value = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))
    return psi_value


In [ ]:
print("Carregando dados Gold e o Modelo de Março...")

df_spark = spark.table("workspace.bacen_mlops.call_center_features")
df_pd = df_spark.toPandas().fillna(0)

# Carrega o modelo de produção
model_uri = "models:/workspace.bacen_mlops.BacenRiskXGBoost/latest"
loaded_model = mlflow.xgboost.load_model(model_uri)

features = [c for c in df_pd.columns if c not in ("call_id", "customer_id", "call_date", "call_month", "target_bacen")]

# Extraindo o Baseline (Q1: Jan, Fev, Mar)
df_baseline = df_pd[df_pd["call_month"] <= 3]
baseline_scores = loaded_model.predict_proba(df_baseline[features])[:, 1]

print(f"Baseline PSI criado! População de Treino: {len(df_baseline)} ligações.")

In [ ]:
print("Simulando a passagem do tempo e medindo o Data Drift...")
meses = [4, 5, 6, 7, 8, 9]
nomes = {4: "Abr", 5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago", 9: "Set"}

psi_history = []

for mes in meses:
    df_mes = df_pd[df_pd["call_month"] == mes]
    if len(df_mes) == 0:
        continue
        
    mes_scores = loaded_model.predict_proba(df_mes[features])[:, 1]
    psi = calculate_psi(baseline_scores, mes_scores)
    
    psi_history.append({'Mês': nomes[mes], 'PSI': psi})
    
    status = "🟢 Estável" if psi < 0.1 else "🟡 Atenção" if psi < 0.2 else "🔴 DRIFT CRÍTICO!"
    print(f"Mês: {nomes[mes]} | Amostras: {len(df_mes)} | PSI: {psi:.4f} -> {status}")

df_historico = pd.DataFrame(psi_history)
df_historico.plot(x='Mês', y='PSI', kind='bar', color=['green' if x < 0.1 else 'orange' if x < 0.2 else 'red' for x in df_historico['PSI']], title="Evolução do Data Drift (PSI)")
plt.axhline(y=0.2, color='r', linestyle='--', label='Limiar Crítico')
plt.legend()
plt.show()